In [1]:
# Challenge 1 Group #7: Regression Problem - Aprtment Rental Price Prediction
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)

In [2]:
# loading datasets
train = pd.read_csv("challenge1_train.csv")
test = pd.read_csv("challenge1_test.csv")

train.head()

,ID,price,bathrooms,bedrooms,square_feet,latitude,longitude,category,has_photo,state
0,2,925,1.0,0,116,47.6160,-122.3275,housing/rent/apartment,Thumbnail,WA
1,3,2475,1.0,0,130,40.7629,-73.9885,housing/rent/apartment,Thumbnail,NY
2,5,1695,1.0,0,190,37.7599,-122.4379,housing/rent/apartment,Thumbnail,CA
3,6,1560,1.0,1,200,35.0847,-77.0609,housing/rent/apartment,Thumbnail,NC
4,7,1560,1.0,1,200,35.0960,-77.0272,housing/rent/apartment,Thumbnail,NC


In [3]:
X = train.drop(columns=["price", "ID"]).copy()
y = train["price"].copy()

# seperating training IDs
train_ids = train["ID"].copy()

# preparing test inputs and preserving IDs
X_test = test.drop(columns=["ID"]).copy()
test_ids = test["ID"].copy()

X.head()

,bathrooms,bedrooms,square_feet,latitude,longitude,category,has_photo,state
0,1.0,0,116,47.6160,-122.3275,housing/rent/apartment,Thumbnail,WA
1,1.0,0,130,40.7629,-73.9885,housing/rent/apartment,Thumbnail,NY
2,1.0,0,190,37.7599,-122.4379,housing/rent/apartment,Thumbnail,CA
3,1.0,1,200,35.0847,-77.0609,housing/rent/apartment,Thumbnail,NC
4,1.0,1,200,35.0960,-77.0272,housing/rent/apartment,Thumbnail,NC


In [4]:
# Pipelines for Numerical and Categorical columns
numeric_features = [
    "bathrooms",
    "bedrooms",
    "square_feet",
    "latitude",
    "longitude"
]

categorical_features = [
    "category",
    "has_photo",
    "state"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse_output=False
    ))
])



In [5]:
# Connecting Pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_na

In [6]:
# checking for every predictor
assert set(numeric_features + categorical_features) == set(X.columns)

assert X.columns.equals(X_test.columns)

print("Preprocessing setup is ready")
print("Training inputs:", X.shape)
print("Training target:", y.shape)
print("Test inputs:", X_test.shape)

Preprocessing setup is ready
Training inputs: (7398, 8)
Training target: (7398,)
Test inputs: (2467, 8)


## Validation Strategy

In [7]:
## continued imports

import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error

In [8]:
## seed & safety check

RANDOM_STATE = 42
assert "ID" not in X.columns and "price" not in X.columns

In [9]:
## hold out split

price_bins = pd.qcut(y, q=10, duplicates="drop")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=price_bins
)

print("Train:", X_train.shape, "| Hold-out:", X_val.shape)

Train: (5918, 8) | Hold-out: (1480, 8)


In [10]:
## checking the split

pd.DataFrame({"train": y_train.describe(), "hold-out": y_val.describe()})

,train,hold-out
count,5918.000000,1480.000000
mean,1460.599020,1471.628378
std,813.573758,853.086628
min,200.000000,288.000000
25%,950.000000,930.000000
50%,1273.500000,1273.500000
75%,1694.750000,1695.000000
max,8800.000000,8500.000000


In [11]:
## folds and scoring

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae":  "neg_mean_absolute_error",
    "r2":   "r2",
}

print([(len(tr), len(va)) for tr, va in cv.split(X_train)])

[(4734, 1184), (4734, 1184), (4734, 1184), (4735, 1183), (4735, 1183)]


In [12]:
## defining functions

def cv_evaluate(name, model):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    res = cross_validate(pipe, X_train, y_train, cv=cv,
                         scoring=scoring, return_train_score=True)
    cv_rmse = -res["test_rmse"].mean()
    train_rmse = -res["train_rmse"].mean()
    return {
        "model": name,
        "CV RMSE (mean)": cv_rmse,
        "CV RMSE (std)": res["test_rmse"].std(),
        "Train RMSE": train_rmse,
        "Gap (CV - Train)": cv_rmse - train_rmse,
        "CV MAE": -res["test_mae"].mean(),
        "CV R2": res["test_r2"].mean(),
    }

def compare(models):
    rows = [cv_evaluate(n, m) for n, m in models.items()]
    return pd.DataFrame(rows).sort_values("CV RMSE (mean)").reset_index(drop=True)

def holdout_rmse(model):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    return np.sqrt(mean_squared_error(y_val, pipe.predict(X_val)))

In [13]:
## final baseline checks

baseline = compare({"Baseline (mean price)": DummyRegressor(strategy="mean")})
print("Std of price in X_train:", y_train.std())
baseline

/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Std of price in X_train: 813.5737582900422


,model,CV RMSE (mean),CV RMSE (std),Train RMSE,Gap (CV - Train),CV MAE,CV R2
0,Baseline (mean price),812.320176,45.528834,813.408304,-1.088128,546.526106,-0.000683


### How to use this validation setup
- Evaluate models with `compare({"Name": model, ...})`; give every model `random_state=RANDOM_STATE`.

- Grid search: `GridSearchCV(Pipeline([("prep", preprocessor), ("model", ...)]), param_grid={"model__<param>": [...]}, cv=cv, scoring="neg_root_mean_squared_error")`, fit on `X_train, y_train` only.
  
- Use `X_val` / `y_val` only once, via `holdout_rmse(final_model)`, after the final model is chosen.
  
- Before predicting `X_test`, refit the final model on all of `X, y`.

## Model Imports

In [15]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}
 
results = compare(models)
results

/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: F

,model,CV RMSE (mean),CV RMSE (std),Train RMSE,Gap (CV - Train),CV MAE,CV R2
0,Hist Gradient Boosting,411.731757,17.843136,310.929528,100.802229,257.115508,0.741832
1,Random Forest,426.928956,17.668385,160.926965,266.001991,261.523614,0.722230
2,Gradient Boosting,454.064703,23.426798,391.865331,62.199373,292.082846,0.687132
3,Linear Regression,518.504486,32.725760,510.861495,7.642991,336.079274,0.592315
4,Decision Tree,563.900533,14.378524,36.987003,526.913530,338.182132,0.513478


## Interpretation

- Hist Gradien Boosting was the best model with the lowest CV RMSE at about (412)
- As tree-based models outperformed linar and single tree models, it's safe to say that the data relationship between prices, features and locations are non-linar.